# import

In [ ]:
import pandas as pd
import random
from utils.recbole_train_test import *
from utils.plot_utils import *
# from utils.model_utils import get_trainer

from datetime import datetime, timezone

from utils.generate_artificial_random_dataset import save_complete_dataset_atomic_file

In [ ]:
SAVE_PATH, BASE_FILENAME, SPECS_STR = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        'NP_ND')


BASE_DATASET_NAME = BASE_FILENAME+'_'+SPECS_STR

# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
MODEL_VERSIONS = d_keys[:duration]


K = [1, 10, 20]
VALID_METRIC = 'Recall@'+str(K[21])
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = True

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

# FILENAME_VERSION = '_ET_ND_LS.t_UD_SF_TO_UM.100'

data_types_dict = {'item_id':int,
                   'user_id':object,
                   'timestamp':float}

In [ ]:
# dataset_pt1_df = pd.DataFrame(dataset_pt1.id2token(dataset_pt1.iid_field, dataset_pt1.inter_feat.interaction[dataset_pt1.iid_field]), columns=['item_id'])
# dataset_pt1_df.item_id = dataset_pt1_df.item_id.astype(int)
# dataset_pt1_df.item_id

def get_id2token(dataset, col='item_id'):
    if col=='item_id':
        return dataset.id2token(dataset.iid_field, dataset.inter_feat.interaction[dataset.iid_field])
    
    elif col=='user_id':
        return dataset.id2token(dataset.uid_field, dataset.inter_feat.interaction[dataset.uid_field])

    elif col=='timestamp':  
        return dataset.id2token(dataset.time_field, dataset.inter_feat.interaction[dataset.time_field])

def recbole_ds_column_2_dataframe(dataset, col='item_id', data_type=int):
    # dataset_df = None
    # if col=='item_id':
    #     dataset_df = pd.DataFrame(get_id2token(dataset, col=col), columns=[col])
    #     if int_type: dataset_df.item_id = dataset_df.item_id.astype(int)
    #     return dataset_df
    
    # elif col=='user_id':
    #     dataset_df = pd.DataFrame(get_id2token(dataset, col=col), columns=[col])
    #     if int_type: dataset_df.user_id = dataset_df.user_id.astype(int)

    # elif col=='timestamp':
    #     dataset_df = pd.DataFrame(get_id2token(dataset, col=col), columns=[col])
    #     if int_type: dataset_df.user_id = dataset_df.user_id.astype(int)

    dataset_df = pd.DataFrame(get_id2token(dataset, col=col), columns=[col])
    dataset_df[col] = dataset_df[col].astype(data_type)
    return dataset_df
        


def recbole_dataset_2_dataframe(dataset, data_types_dict):
    i = recbole_ds_column_2_dataframe(dataset, col='item_id', data_type=data_types_dict['item_id'])
    u = recbole_ds_column_2_dataframe(dataset, col='user_id', data_type=data_types_dict['user_id'])
    # t = recbole_ds_column_2_dataframe(dataset, col='timestamp', data_type=data_types_dict['timestamp'])

    df = pd.DataFrame({'user_id': u.user_id,
                         'item_id': i.item_id})
                        #  'timestamp': dataset.inter_feat.interaction[dataset.time_field]})

    return df
    

In [ ]:
def get_list_users_with_only_1_inter_in_at_least_one_quarter(df_inter):
    df_inter['date'] = df_inter['timestamp'].apply(lambda x: datetime.fromtimestamp(x, timezone.utc)) 
    df_inter["quarter"] = df_inter["date"].dt.to_period("Q")  # Extract quarter-year
    quarter_inter = df_inter[['user_id', 'quarter']].groupby(['quarter', 'user_id']).size()
    quarter_inter = quarter_inter.reset_index(name='n_inter')

    list_users_1_interaction = list(quarter_inter.loc[quarter_inter['n_inter']==1, 'user_id'].unique())
    print('number of users with 1 interaction in at least one of the quarters they\'re active:',len(list_users_1_interaction))
    return list_users_1_interaction


def sample_out_users_1_inter(df_inter, list_users_1_interaction):
    sampled_df = df_inter.loc[~df_inter.user_id.isin(list_users_1_interaction)].reset_index(drop=True)
    
    # just to verify it worked
    # _sdf = sampled_df[['user_id', 'quarter']].groupby(['quarter', 'user_id']).size().reset_index(name='n_inter')
    # print('is it empty? :', _sdf.loc[_sdf['n_inter']==1, 'quarter'].value_counts().sort_index()) # it's supposed to be empty
    try:
        stats = {
        'Number of users' : sampled_df.user_id.nunique(),
        'Number of items' : sampled_df.item_id.nunique(),
        'Total number of interactions' : [sampled_df.shape[0], sampled_df[['item_id']].describe()],
        'Mean of interactios per user' : sampled_df[['user_id', 'item_id']].groupby(by='user_id').mean()

        }
    except:
        stats = {
        'Number of users' : sampled_df.user_id.nunique(),
        'Number of items' : sampled_df.item_id.nunique(),
        'Total number of interactions' : [sampled_df.shape[0], sampled_df[['item_id']].describe()],
        # 'Mean of interactios per user' : sampled_df[['user_id', 'item_id']].groupby(by='user_id').mean()
        }

    print('\n   Statistics:\n')
    for stat, value in stats.items():
        if type(value) is list:
            print(f'{stat}:')
            for v in value:
                print(f'{v}')
            print('\n')
        else:
            print(f'{stat}:\n{value}\n\n')

    return sampled_df


this approach removes all user-item interactions that appear in the test set. So, it will remove the re-occurence of user-item. This challenges the requirenment of users with more than 2 interaction in each quarter they're active.
so, start with pt4, extract all users that respect the requirenments. Filter the smaller models regarding the users and the testset user-item interactions.

In [ ]:
model_name = 'BPR'

# from complete dataset (aka pt4)
part = MODEL_VERSIONS[-1]

dataset_name = BASE_DATASET_NAME+part
df_pt4 = pd.read_csv(SAVE_PATH+dataset_name+'/'+dataset_name+'.csv')
# print('start:',model_df_pti.shape)

test_data_sections = get_test_data_sections_with_names(model_version=part,
                                                        base_dataset_name=BASE_DATASET_NAME,
                                                        models_versions=MODEL_VERSIONS)[:-1]#[:1]


print(part, df_pt4.shape)
start_rows = df_pt4.shape[0]

# remove all sectioned parts' test sets
for sectioned_pti_name in test_data_sections:
    
    parameter_dict = {
        'dataset': sectioned_pti_name+'.inter',
        'use_gpu':USE_GPU,

        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': SAVE_PATH,
        'save_dataset':False,# save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':SAVE_PATH+sectioned_pti_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    _, _, _, _, _, test_data_pti = setup_config_and_dataset(model_name, sectioned_pti_name, parameter_dict)

    test_pti_df = recbole_dataset_2_dataframe(test_data_pti._dataset, data_types_dict)

    # remove section testset (will remove all instances of those interactions, there will be no re-occurences)
    df_pt4 = df_pt4.merge(test_pti_df, on=['user_id', 'item_id'], how='left', indicator=True)
    df_pt4 = df_pt4[df_pt4['_merge'] == 'left_only'].drop(columns=['_merge'])
    print(part, sectioned_pti_name, df_pt4.shape[0]/start_rows)

print('cleaned of testsets:',df_pt4.shape[0]/start_rows)

# to guarantee that there will be a testset in all parts for the active users
# remove users that only have 1 interaction in at least one quarter
list_users_1_inter = get_list_users_with_only_1_inter_in_at_least_one_quarter(df_pt4)
sampled_df_pt4 = sample_out_users_1_inter(df_pt4, list_users_1_inter)

# time interval

In [ ]:
def interval_list_as_timestamp(split_intervals):
    '''
        example:
        split_intervals = [ ['2012-07','2012-12'],
                            ['2013-01','2013-06'],
                            ['2013-07','2013-12'],
                            ['2014-01','2014-06'],
                            ['2014-07','2015-01']
                        ]
    '''
    return [[pd.Timestamp(interval[0]), pd.Timestamp(interval[1])] for interval in split_intervals]


split_intervals_list = interval_list_as_timestamp( [ ['2012-07','2012-12'], # cold start bucket
                                               
                                                ['2013-01','2013-06'], # pt 1
                                                ['2013-07','2013-12'], # pt 5
                                                ['2014-01','2014-06'], # pt 6, shift starts
                                                ['2014-07','2015-01'], # pt 7

                                                ['2013-01','2013-12'], # pt 2
                                                ['2013-01','2014-06'], # pt 3
                                                ['2013-01','2015-01'], # pt 4

                                                ['2013-07','2014-06'] # pt 8
                                                ]
                                            )
d_keys = ['cs','pt1','pt5','pt6','pt7','pt2','pt3','pt4','pt8']
split_intervals_dict = { d_keys[i]: ts for i, ts in enumerate(split_intervals_list)}

# Introduce Random Drift (or shift, bc it's sudden) in data

## variable sudden drift start

In [ ]:
train_buckets_shift = sampled_df_pt4.copy()

In [ ]:
sudden_drift_start = train_buckets_shift[(train_buckets_shift.date >= split_intervals_dict['pt6'][0])].index[0]
sudden_drift_start

## functions
these functions need to be locally defined bc they are used in apply functions and use global variables

In [ ]:
# def calculate_sparsity(df):
#         # df.item_id.groupby([df.user_id, df.item_id]).count().sum() == df.user_id.count()
#         sparsity = 1 - df.user_id.count()/(df.user_id.nunique()*df.item_id.nunique())
#         specs_str = str(df.user_id.nunique())+'x'+str(df.item_id.nunique())+'_'+str(round(sparsity, 2))
#         print('specs_str', specs_str)
#         return sparsity, specs_str
from utils.data_utils import *
    

def rename_item(row):
    global sudden_drift_start
    global renamed_items
    
    # print('sudden_drift_start=', sudden_drift_start)
    # print('renamed_items=', renamed_items)

    if int(row.name) > sudden_drift_start and row['item_id'] in renamed_items:
        return renamed_items[row['item_id']]
    return row['item_id']

## variables

In [ ]:
items_list = list(train_buckets_shift.item_id.unique())
users_list = list(train_buckets_shift.user_id.unique())

n_users = train_buckets_shift.user_id.nunique()
n_items = train_buckets_shift.item_id.nunique()
n_interactions = train_buckets_shift.shape[0]

random_seed = 42

n_items_to_drift = n_items // 2 # 50% 

random.seed(random_seed)  # For reproducibility
drift_items_list = random.sample(items_list, k=n_items_to_drift)  

# select the most popular items to drift
# drift_items_list = list(train_buckets.item_id.value_counts().index)[:n_items_to_drift]
# drift_items_list

renamed_items = {item: f'd_{item}' for item in drift_items_list}
non_drift_items_list = list(set(items_list) - set(drift_items_list))

## introduce shift (rename function)

In [ ]:
train_buckets_shift['item_id'] = train_buckets_shift.apply(rename_item, axis=1)

sparsity, specs_str = calculate_sparsity(train_buckets_shift)
print('sparsity: ',sparsity)
# print(train_buckets_shift.item_id.groupby([train_buckets_shift.user_id, train_buckets_shift.item_id]).count().unstack().fillna(0).astype(int))

# save RD data: split in 4, 6 month each

NP_NT_RD

No Pre-Train set (Cold start + User Items drifted_items Initialised),
No testset in trainset,
Random Drift

In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df',
                                        'NPT_NT_RD.'+(n_items_to_drift/n_items*100))

for pti in MODEL_VERSIONS:
    print(pti)
    time_interval = split_intervals_dict[pti]
    
    df=sampled_df_pt4[(train_buckets_shift.date >= time_interval[0]) & (train_buckets_shift.date <= time_interval[1])]
    print('\nnumber of users: ')
    print(df.user_id.nunique())
    print('\nnumber of items: ')
    print(df.item_id.nunique())
    print('\ntotal number of interactions: ')
    print(df.shape[0])
    print(df[['item_id']].describe())
    # print('\nmean of interactios per user: ')
    # print(df_inter[['user_id', 'item_id']].describe())
    # print(df[['user_id', 'item_id']].groupby(by='user_id').mean())
    
    # cs_df = pd.concat([cold_start_bucket, df])
    save_complete_dataset_atomic_file(df=df,
                                      save_path=save_path,
                                      base_filename=base_filename,
                                      specs_str=specs_str+'_'+pti)